<a href="https://colab.research.google.com/github/Richan-A27/Project_RADIX/blob/main/Talent_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install required library for reading Excel files silently
!pip install -q openpyxl pandas

import json
import pandas as pd
from google.colab import drive

# Mount Google Drive to access the spreadsheet
drive.mount('/content/drive', force_remount=True)

# Competency Tier Matrix
COMPETENCY_TIERS = {
    "0-NONE": 0, "2-CU": 1, "3-CU": 2, "4-CU": 3, "4-AP": 4,
    "5-AP": 5, "6-AP": 6, "5-AS": 7, "6-AS": 8, "7-AS": 9,
    "8-AS": 10, "9-EV": 11
}

def run_talent_check(candidate_json_str, company_name, spreadsheet_path):
    try:
        candidate_data = json.loads(candidate_json_str)
    except json.JSONDecodeError:
        return json.dumps({"error": "Invalid Candidate JSON provided."})

    try:
        # Load the Skill Levels spreadsheet
        df = pd.read_excel(spreadsheet_path)
        df.columns = df.columns.str.strip().str.lower()
    except Exception as e:
        return json.dumps({"error": f"Failed to load spreadsheet: {e}"})

    # Check if company exists
    company_row = df[df['companies'].str.lower() == company_name.lower()]
    if company_row.empty:
        return json.dumps({"error": f"Company '{company_name}' not found in the benchmark database."})

    company_row = company_row.iloc[0]

    # Map candidate skills
    candidate_skills = {
        skill.get('category_code', '').strip().lower(): skill.get('candidate_level', '0-NONE')
        for skill in candidate_data.get('skills', [])
    }

    expected_categories = [
        "coding", "data_structures_and_algorithms", "object_oriented_programming_and_design",
        "aptitude_and_problem_solving", "communication_skills", "ai_native_engineering"
    ]

    skillset_gap = []
    passed_categories = 0

    for category in expected_categories:
        required_str = str(company_row.get(category, "0-NONE")).strip()
        candidate_str = candidate_skills.get(category, "0-NONE").strip()

        req_val = COMPETENCY_TIERS.get(required_str, 0)
        cand_val = COMPETENCY_TIERS.get(candidate_str, 0)

        is_gap = cand_val < req_val
        if not is_gap:
            passed_categories += 1

        skillset_gap.append({
            "category_code": category,
            "required_level": required_str,
            "candidate_level": candidate_str,
            "gap": is_gap
        })

    readiness_score = int((passed_categories / len(expected_categories)) * 100)

    output_json = {
        "company": company_row.get('companies', company_name),
        "readiness_score": readiness_score,
        "skillset_gap": skillset_gap
    }

    return json.dumps(output_json, indent=2)

# ==========================================
# TEST EXECUTION AREA
# ==========================================
mock_candidate_profile = """
{
  "name": "Jane Doe",
  "email": "jane.doe@example.com",
  "skills": [
    {"skill_name": "Python", "category_code": "coding", "candidate_level": "8-AS"},
    {"skill_name": "Trees/Graphs", "category_code": "data_structures_and_algorithms", "candidate_level": "7-AS"},
    {"skill_name": "Design Patterns", "category_code": "object_oriented_programming_and_design", "candidate_level": "6-AP"},
    {"skill_name": "Puzzles", "category_code": "aptitude_and_problem_solving", "candidate_level": "6-AS"},
    {"skill_name": "Technical Presentation", "category_code": "communication_skills", "candidate_level": "7-AS"}
  ]
}
"""

SPREADSHEET_FILE = "/content/skilll levels company.xlsx"
TARGET_COMPANY = "Barclays PLC"

result = run_talent_check(mock_candidate_profile, TARGET_COMPANY, SPREADSHEET_FILE)
print(result)

Mounted at /content/drive
{
  "company": "Barclays PLC",
  "readiness_score": 66,
  "skillset_gap": [
    {
      "category_code": "coding",
      "required_level": "7-AS",
      "candidate_level": "8-AS",
      "gap": false
    },
    {
      "category_code": "data_structures_and_algorithms",
      "required_level": "7-AS",
      "candidate_level": "7-AS",
      "gap": false
    },
    {
      "category_code": "object_oriented_programming_and_design",
      "required_level": "7-AS",
      "candidate_level": "6-AP",
      "gap": true
    },
    {
      "category_code": "aptitude_and_problem_solving",
      "required_level": "6-AS",
      "candidate_level": "6-AS",
      "gap": false
    },
    {
      "category_code": "communication_skills",
      "required_level": "7-AS",
      "candidate_level": "7-AS",
      "gap": false
    },
    {
      "category_code": "ai_native_engineering",
      "required_level": "4-CU",
      "candidate_level": "0-NONE",
      "gap": true
    }
  ]
}
